# ¿Dónde miran las cabezas cuando el atributo es *Attractive*? (CelebA, 2026-11-02)

**Por qué este atributo.** Casi todos los atributos de CelebA tienen una señal visible y localizable: `Eyeglasses` está en los ojos, `Wearing_Hat` arriba, `Mouth_Slightly_Open` en la boca, `Gray_Hair` en el pelo. **`Attractive` es subjetivo**: no hay una región de la cara que "sea" atractiva. Es un buen caso de contraste para preguntar si lo que vimos en `celeba_regions.ipynb` (cabezas penalizadas que se dedican a ojos, boca, pelo) sirve para algo más que atributos con una posición evidente.

**Qué se pregunta (en este orden):**
1. **¿Dónde mira cada cabeza** en imágenes `Attractive = 1` frente a `Attractive = 0`? ¿Cambia el patrón? ¿Cambia *menos* que en un atributo objetivo (gafas, sombrero, boca abierta)?
2. **¿Se puede predecir `Attractive` solo a partir de *dónde mira* la cabeza?** (una sonda lineal sobre el mapa de atención de 256 parches). Comparado con los otros 39 atributos.
3. **¿Eso es "atractivo" o son sus correlatos?** `Attractive` está muy correlacionado con `Male`, `Young`, `Heavy_Makeup`, `Wearing_Lipstick`, `Smiling`... Si la atención sólo codifica esos, el resultado sobre `Attractive` es un espejismo. Lo controlo estratificando por `Male × Young`.
4. **¿Cuánto pesa cada cabeza en la predicción de `Attractive`?** (sondas sobre la salida de cada cabeza + **ablación**: quitar la cabeza y ver cuánto cae la predicción). Con la región que mira cada cabeza, esto permite decir **"el modelo decide `Attractive` apoyándose en la cabeza que mira X"**: la parte de explicabilidad del notebook.
5. **Los 3 modelos × 3 semillas** (`unconstrained`, `constrained`, `constrained_rank`): ¿el patrón es de la penalización o de cada entrenamiento?

> **Antes de leer los resultados:** `Attractive` es una etiqueta humana subjetiva y sesgada (en CelebA los anotadores la asocian a juventud, maquillaje y género femenino). Aquí **no** se evalúa a ninguna persona ni se afirma qué es atractivo: se estudia qué estadísticos de la imagen usa un modelo para reproducir una etiqueta ruidosa.

## Glosario (lo nuevo respecto a `celeba_regions.ipynb`)

- **Masa en una región** y **enriquecimiento**: igual que en `celeba_regions.ipynb` (cuánta atención del [CLS] cae en ojos, nariz, boca, cara, fuera de la cara).
- **Diferencia de medias (Attractive=1 − Attractive=0)** de un mapa de atención: dónde mira *más* cuando la etiqueta es 1.
- **d de Cohen**: diferencia de medias dividida por la desviación típica combinada. |d| < 0.2 ≈ diferencia pequeña, 0.5 ≈ mediana, ≥ 0.8 ≈ grande. Con ~20 000 imágenes todo sale "estadísticamente significativo", por eso uso el tamaño del efecto y no el p-valor.
- **AUC**: probabilidad de que una imagen con etiqueta 1 reciba un puntaje mayor que una con etiqueta 0. 0.5 = azar, 1 = perfecto.
- **AUC estratificado**: el AUC calculado *dentro* de cada grupo (Male × Young) y promediado. Si un predictor sólo hace trampa separando hombres de mujeres o jóvenes de mayores, dentro del grupo el AUC cae a ~0.5.

## 0. Configuración

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

project_root = Path.cwd().resolve().parents[1]
sys.path.append(str(project_root))
from modules.figures import save_fig
from modules.model import ViTForClassfication

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
EXPERIMENTS_DIR = project_root / "experimentation" / "experiments"
OUTPUT_DIR = EXPERIMENTS_DIR / "02112026_celeba_attractive"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE = project_root / "data" / "celeba" / "celeba_test64.npz"   # la crea celeba_regions.ipynb

VARIANTS = ["unconstrained", "constrained", "constrained_rank"]
SEEDS = [0, 1, 2]
BLOCK, N_HEADS, GRID = 4, 4, 16
TARGET = "Attractive"
CONTEXT = ["Eyeglasses", "Wearing_Hat", "Mouth_Slightly_Open", "Male", "Smiling"]   # atributos objetivos/ locales para comparar
SEED = 0
print("device:", device)

device: mps


## 1. Datos y qué tan "especial" es `Attractive`

Se usa la **misma caché de test** que `celeba_regions.ipynb` (19 962 imágenes de 64×64, 40 atributos y 5 landmarks por imagen). Si no existe, corre primero ese notebook, que la construye.

In [2]:
if not CACHE.exists():
    raise FileNotFoundError("Falta data/celeba/celeba_test64.npz: ejecuta primero notebooks/02112026_celeba_regions/celeba_regions.ipynb")
D = np.load(CACHE, allow_pickle=True)
X8, ATTR, LM = D["x"], D["attr"], D["lm"]
ATTR_NAMES = list(D["attr_names"]); N = len(X8)
IA = ATTR_NAMES.index(TARGET)
y = ATTR[:, IA]
print(f"{N} imágenes | {TARGET} = 1 en {y.mean():.1%} (tasa base)")

# Con qué otros atributos va junta Attractive (correlación phi = Pearson entre variables 0/1)
corr = pd.Series(np.corrcoef(ATTR.T.astype(float))[IA], index=ATTR_NAMES).drop(TARGET)
print("\nAtributos más correlacionados con Attractive:")
display(corr.reindex(corr.abs().sort_values(ascending=False).index).head(10).round(2).to_frame("correlación"))

19962 imágenes | Attractive = 1 en 49.6% (tasa base)

Atributos más correlacionados con Attractive:


,correlación
Wearing_Lipstick,0.47
Heavy_Makeup,0.47
Male,-0.39
Young,0.38
Arched_Eyebrows,0.27
Pointy_Nose,0.26
Big_Nose,-0.24
Chubby,-0.22
Eyeglasses,-0.22
Wavy_Hair,0.21


Lo que ves arriba es la clave para leer todo el notebook: `Attractive` no es una propiedad visual propia, sino que **va con** maquillaje, labial, juventud y género. Por eso la pregunta 3 (control por correlatos) no es opcional.

## 2. Regiones (mismas definiciones que en `celeba_regions.ipynb`)

Sobre la rejilla de 16×16 parches: `ojos`, `nariz`, `boca` = parches a ≤ 5 px de los landmarks (en 64×64); `cara` = caja de los landmarks + 6 px; `fuera de la cara` = el resto (pelo, fondo); `frente/pelo` = más de 7 px por encima de los ojos.

In [3]:
centros = (np.arange(GRID) * 4 + 2).astype(np.float32)
PX, PY = np.meshgrid(centros, centros); PX, PY = PX.ravel()[None], PY.ravel()[None]
disco = lambda c, r: ((PX - c[:, 0:1]) ** 2 + (PY - c[:, 1:2]) ** 2) <= r ** 2
def segmento(a, b, r):
    ab = b - a
    t = (((PX - a[:, 0:1]) * ab[:, 0:1] + (PY - a[:, 1:2]) * ab[:, 1:2]) / (ab ** 2).sum(1, keepdims=True)).clip(0, 1)
    return ((PX - (a[:, 0:1] + t * ab[:, 0:1])) ** 2 + (PY - (a[:, 1:2] + t * ab[:, 1:2])) ** 2) <= r ** 2
oi, od, no, bi, bd = (LM[:, k] for k in range(5))
x0, x1, y0, y1 = LM[..., 0].min(1) - 6, LM[..., 0].max(1) + 6, LM[..., 1].min(1) - 6, LM[..., 1].max(1) + 6
cara = (PX >= x0[:, None]) & (PX <= x1[:, None]) & (PY >= y0[:, None]) & (PY <= y1[:, None])
REGIONES = {"ojos": disco(oi, 5) | disco(od, 5), "nariz": disco(no, 5), "boca": segmento(bi, bd, 5),
            "cara": cara, "fuera de la cara": ~cara, "frente/pelo": PY < ((oi[:, 1] + od[:, 1]) / 2)[:, None] - 7}
AREA = {k: m.mean(1) for k, m in REGIONES.items()}
print({k: round(float(v.mean()), 3) for k, v in AREA.items()})

{'ojos': 0.037, 'nariz': 0.019, 'boca': 0.045, 'cara': 0.152, 'fuera de la cara': 0.848, 'frente/pelo': 0.389}


## 3. Extraer atención, salidas de cada cabeza y logits tras quitar cada cabeza

Para cada modelo, sobre las 19 962 imágenes de test, guardo:
- **`att`**: atención del [CLS] a los 256 parches (bloque 4, por cabeza, suma 1).
- **`z`**: la **salida de cada cabeza** en el [CLS] (12 números por cabeza, antes de mezclarlas en la proyección `W_O`). Es lo que usan las sondas de la sección 9.
- **`logit`**: los 40 logits del modelo completo.
- **`logit_abl[h]`**: los 40 logits **después de ablar la cabeza h**, es decir, reemplazar su salida en el [CLS] por su **valor medio** sobre todas las imágenes (así el resto del modelo no ve un valor "raro" como pasaría con ceros). Solo se re-ejecuta el bloque 4 y el clasificador, no todo el modelo.

Una verificación (`assert`) confirma que esta ejecución por partes reproduce los logits del modelo normal.

In [ ]:
def cargar_modelo(nombre):
    carpeta = EXPERIMENTS_DIR / nombre
    m = ViTForClassfication(json.loads((carpeta / "config.json").read_text()))
    m.load_state_dict(torch.load(carpeta / "model_final.pt", map_location="cpu"))
    return m.to(device).eval()


@torch.no_grad()
def analizar(modelo, lote=100):
    """Atención, salida por cabeza y logits (completo y con cada cabeza ablada) de un modelo sobre todo el test."""
    blk = modelo.encoder.blocks[BLOCK - 1]
    proj = blk.attention.output_projection
    hs = blk.attention.attention_head_size                              # 12 números por cabeza
    imgs = lambda i: torch.tensor(X8[i:i + lote]).permute(0, 3, 1, 2).float().div(255).sub(0.5).div(0.5).to(device)

    # Pasada 1: llevar todas las imágenes hasta la ENTRADA del bloque 4 (se guarda en CPU y se reutiliza para las ablaciones)
    X3 = []
    for i in range(0, N, lote):
        h = modelo.embedding(imgs(i))
        for b in modelo.encoder.blocks[:BLOCK - 1]:
            h, _ = b(h)
        X3.append(h.cpu())
    X3 = torch.cat(X3)

    # Pasada 2: bloque 4 + clasificador, capturando la salida de las cabezas (entrada de la proyección W_O)
    cap = {}
    hook = proj.register_forward_pre_hook(lambda m, args: cap.__setitem__("z", args[0][:, 0].detach().clone()))
    atts, zs, logs = [], [], []
    for i in range(0, N, lote):
        out, att = blk(X3[i:i + lote].to(device), output_attentions=True)
        logs.append(modelo.classifier(out[:, 0]).cpu()); zs.append(cap["z"].cpu())
        atts.append(att[:, :, 0, 1:].clone().cpu())                      # CLS -> parches
    hook.remove()
    logit, z = torch.cat(logs), torch.cat(zs)                            # (N,40), (N,48)
    # Verificación: reproduce el modelo normal
    chk, _ = modelo(imgs(0))
    assert torch.allclose(chk.cpu(), logit[:lote], atol=1e-3), "la ejecución por partes no coincide con el modelo"

    # Pasada 3: ablación de cada cabeza (se reemplaza su salida en el [CLS] por su media sobre las imágenes)
    z_medio = z.mean(0)
    abl = []
    for h in range(N_HEADS):
        def pre(m, args, h=h):
            zz = args[0].clone(); zz[:, 0, h * hs:(h + 1) * hs] = z_medio[h * hs:(h + 1) * hs].to(zz.device)
            return (zz,)
        hk = proj.register_forward_pre_hook(pre)
        ls = [modelo.classifier(blk(X3[i:i + lote].to(device))[0][:, 0]).cpu() for i in range(0, N, lote)]
        hk.remove(); abl.append(torch.cat(ls))
    # Ablación de LAS CUATRO cabezas a la vez (el bloque 4 sin su atención en el [CLS])
    def pre_all(m, args):
        zz = args[0].clone(); zz[:, 0] = z_medio.to(zz.device); return (zz,)
    hk = proj.register_forward_pre_hook(pre_all)
    ls = [modelo.classifier(blk(X3[i:i + lote].to(device))[0][:, 0]).cpu() for i in range(0, N, lote)]
    hk.remove(); abl_all = torch.cat(ls)
    a = torch.cat(atts).numpy(); a = a / a.sum(-1, keepdims=True)
    return {"att": a.astype(np.float32), "logit": logit.numpy(), "z": z.numpy().reshape(N, N_HEADS, hs),
            "logit_abl": torch.stack(abl).numpy(), "logit_abl_all": abl_all.numpy()}   # (4, N, 40), (N, 40)


ATT, LOGIT, Z, LABL, LALL = {}, {}, {}, {}, {}
for v in VARIANTS:
    for s in SEEDS:
        r = analizar(cargar_modelo(f"vit-5-{v}-s{s}-CelebA"))
        ATT[(v, s)], LOGIT[(v, s)], Z[(v, s)], LABL[(v, s)], LALL[(v, s)] = r["att"], r["logit"], r["z"], r["logit_abl"], r["logit_abl_all"]
        print(f"{v:>17} s{s}: AUC del propio modelo para {TARGET} = {roc_auc_score(y, r['logit'][:, IA]):.3f}", flush=True)

    unconstrained s0: AUC del propio modelo para Attractive = 0.871
    unconstrained s1: AUC del propio modelo para Attractive = 0.862
    unconstrained s2: AUC del propio modelo para Attractive = 0.867
      constrained s0: AUC del propio modelo para Attractive = 0.864
      constrained s1: AUC del propio modelo para Attractive = 0.869


## 4. Pregunta 1 — ¿Cambia *dónde* miran las cabezas entre `Attractive = 1` y `= 0`?

**Cómo leer las figuras de esta sección** (una por variante, semilla 0). Cada figura tiene **dos filas**, una columna por cabeza:

- **Fila de arriba — "Dónde mira en general":** la atención media de la cabeza sobre *todas* las imágenes. Es un mapa de 16×16 parches (la cara de 64×64 píxeles partida en cuadritos). **Más claro/amarillo = mira más ese parche.** Los puntos blancos marcan dónde están, en promedio, los ojos, la nariz y la boca, para orientarte. El título dice cuántos parches usa realmente la cabeza (1 = mira siempre el mismo).
- **Fila de abajo — "Qué cambia cuando la etiqueta es *Attractive*":** `mapa medio con Attractive=1` **menos** `mapa medio con Attractive=0`.
  - 🔴 **Rojo = la cabeza mira MÁS ese parche cuando la persona está etiquetada como *Attractive*.**
  - 🔵 **Azul = la cabeza mira MÁS ese parche cuando la persona *NO* está etiquetada como Attractive** (es decir, mira menos cuando es Attractive).
  - ⚪ **Blanco = no cambia.**
  - La escala de cada panel es propia (se indica en la barra de color): lo importante es **dónde** hay color, no que un panel sea más intenso que otro. Si un panel es casi todo blanco, esa cabeza **no cambia su mirada** con el atributo.

Esto describe *dónde miran*, no *cuánto pesa* esa cabeza en la predicción; eso se mide en las secciones 8 y 9.

In [ ]:
lm_medio = LM.mean(0) / 4 - 0.5                  # posición media de los landmarks en unidades de parche
ETIQ_LM = ["ojo izq.", "ojo der.", "nariz", "boca izq.", "boca der."]

def figura_diferencias(v, s):
    a = ATT[(v, s)]
    fig, axes = plt.subplots(2, N_HEADS, figsize=(14, 7.6))
    for h in range(N_HEADS):
        medio = a[:, h].mean(0).reshape(GRID, GRID)
        dif = (a[y == 1, h].mean(0) - a[y == 0, h].mean(0)).reshape(GRID, GRID)
        eff = float(np.exp(-(a[:, h] * np.log(np.clip(a[:, h], 1e-8, None))).sum(1)).mean())
        top = max(["ojos", "nariz", "boca", "fuera de la cara"], key=lambda k: (a[:, h] * REGIONES[k]).sum(1).mean())
        ax = axes[0, h]; im = ax.imshow(medio, cmap="magma")
        ax.scatter(lm_medio[:, 0], lm_medio[:, 1], c="white", s=14, edgecolors="k", linewidths=0.5)
        ax.set_title(f"CABEZA {h + 1}\nmira sobre todo: {top}\n({eff:.0f} parches efectivos)", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02); cb.set_label("atención media", fontsize=7)
        ax = axes[1, h]; vm = max(np.abs(dif).max(), 1e-6)
        im = ax.imshow(dif, cmap="RdBu_r", vmin=-vm, vmax=vm)
        ax.scatter(lm_medio[:, 0], lm_medio[:, 1], c="k", s=10)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"cambio al ser {TARGET}\n(máx |cambio| = {vm:.3f})", fontsize=9)
        cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02, ticks=[-vm, 0, vm])
        cb.ax.set_yticklabels(["azul: más si\nNO Attractive", "sin\ncambio", "rojo: más si\nAttractive"], fontsize=6.5)
    axes[0, 0].set_ylabel("Dónde mira\n(todas las imágenes)", fontsize=10)
    axes[1, 0].set_ylabel(f"{TARGET}=1 menos {TARGET}=0", fontsize=10)
    fig.suptitle(f"Modelo {v}, semilla {s}, bloque {BLOCK}. Puntos = posición media de ojos, nariz y boca.\n"
                 f"Arriba: dónde mira cada cabeza. Abajo: en qué parches mira MÁS (rojo) o MENOS (azul) cuando la imagen es {TARGET}.", fontsize=11)
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    return fig

for v in VARIANTS:
    fig = figura_diferencias(v, 0)
    save_fig(fig, f"attention_difference_maps_{v}", OUTPUT_DIR, tag="celeba_seed0", project_root=project_root)
    plt.show()

**Tabla: tamaño del efecto por región (d de Cohen).** Para cada modelo, cabeza y región: cuánto difiere la masa de atención entre `Attractive = 1` y `= 0`, en desviaciones típicas. Para dar escala, lo mismo con atributos objetivos (`CONTEXT`). Reporto, por modelo y atributo, el **mayor |d| entre las 4 cabezas × 6 regiones**: es "el cambio de mirada más grande que provoca ese atributo".

In [ ]:
def cohen_d(m, lab):
    a, b = m[lab == 1], m[lab == 0]
    sp = np.sqrt(((len(a) - 1) * a.var(0, ddof=1) + (len(b) - 1) * b.var(0, ddof=1)) / (len(a) + len(b) - 2))
    return (a.mean(0) - b.mean(0)) / np.clip(sp, 1e-9, None)

filas = []
for (v, s), a in ATT.items():
    masa = np.stack([(a * REGIONES[k][:, None, :]).sum(-1) for k in REGIONES], -1)       # (N, 4 cabezas, 6 regiones)
    for atr in [TARGET] + CONTEXT:
        d = np.stack([cohen_d(masa[:, h, :], ATTR[:, ATTR_NAMES.index(atr)]) for h in range(N_HEADS)])   # (4, 6)
        h_, r_ = np.unravel_index(np.nanargmax(np.abs(d)), d.shape)
        filas.append({"modelo": v, "semilla": s, "atributo": atr, "max |d|": float(np.nanmax(np.abs(d))),
                      "cabeza": h_ + 1, "región": list(REGIONES)[r_], "d con signo": float(d[h_, r_])})
EFECTO = pd.DataFrame(filas)
EFECTO.to_csv(OUTPUT_DIR / "attention_shift_effect_sizes.csv", index=False)
piv = EFECTO.pivot_table(index=["modelo", "semilla"], columns="atributo", values="max |d|").loc[VARIANTS][[TARGET] + CONTEXT]
display(piv.round(2))
print("Promedio sobre semillas:"); display(piv.groupby(level=0).mean().loc[VARIANTS].round(2))

**Cómo leerlo:** si `Attractive` tuviera una señal localizada como `Eyeglasses`, su |d| sería comparable al de las gafas en las cabezas de ojos. Si es mucho menor, el atributo **no desplaza hacia ninguna parte** la mirada de las cabezas.

## 5. Pregunta 2 — ¿Se predice `Attractive` solo con *dónde* mira una cabeza?

Para cada modelo y cada cabeza, tomo su mapa de atención (256 números por imagen; aplico √ para que los parches con poca atención pesen algo y estandarizo) y entreno una **sonda lineal** (Ridge) con validación cruzada de 5 folds para predecir los 40 atributos. Mido el AUC fuera de muestra. Esto responde: *¿la información "dónde mira" contiene señal de este atributo?* (no es lo mismo que "la cabeza usa ese atributo").

Para que sea comparable, calculo los 40 atributos y miro en qué lugar queda `Attractive`.

In [ ]:
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(y, y))
Y_pm = ATTR.astype(np.float32) * 2 - 1           # etiquetas -1/+1 (regresión sobre etiquetas = sonda lineal tipo LDA)

def auc_atencion(mapa):
    """mapa (N, 256) -> predicciones fuera de muestra (N, 40) de una sonda lineal sobre √atención."""
    X = np.sqrt(mapa); X = (X - X.mean(0)) / np.clip(X.std(0), 1e-6, None)
    pred = np.zeros_like(Y_pm)
    for tr, te in folds:
        r = RidgeCV(alphas=[10, 100, 1000, 10000], alpha_per_target=True).fit(X[tr], Y_pm[tr])
        pred[te] = r.predict(X[te])
    return pred

PRED = {}                                         # (variante, semilla, cabeza) -> (N, 40) predicciones
filas = []
for (v, s), a in ATT.items():
    for h in range(N_HEADS):
        pr = auc_atencion(a[:, h]); PRED[(v, s, h)] = pr
        aucs = np.array([roc_auc_score(ATTR[:, j], pr[:, j]) for j in range(len(ATTR_NAMES))])
        filas.append({"modelo": v, "semilla": s, "cabeza": h + 1, "AUC Attractive": aucs[IA],
                      "rango de Attractive (1=mejor de 40)": int((aucs > aucs[IA]).sum() + 1),
                      "AUC mediano de los 40": float(np.median(aucs)),
                      **{f"AUC {c}": aucs[ATTR_NAMES.index(c)] for c in CONTEXT}})
AUCT = pd.DataFrame(filas)
AUCT.to_csv(OUTPUT_DIR / "attention_probe_auc.csv", index=False)
display(AUCT.round(3))
print("Promedio por modelo (sobre 3 semillas × 4 cabezas):")
display(AUCT.groupby("modelo")[[c for c in AUCT.columns if c.startswith("AUC") or c.startswith("rango")]].mean().loc[VARIANTS].round(3))

### 5b. Lo mismo, pero sobre la **salida** de cada cabeza (incluye V)

La celda de arriba usa solo el **mapa de atención** (*dónde* mira, 256 números por imagen). Aquí se repite exactamente la misma prueba (sonda Ridge, 40 atributos, mismos 5 folds, AUC fuera de muestra) pero con la **salida `z` de la cabeza** en el [CLS] (12 números por imagen), que es el mapa de atención aplicado a V: *lo que la cabeza entrega*.

Así la comparación es limpia: mismo método, solo cambia la entrada. Si el AUC sube respecto a la sección 5, el contenido (V) aporta información que la mera ubicación de la mirada no tiene. La última tabla pone ambos lado a lado.

In [ ]:
def auc_salida(F):
    """F (N, 12) = salida de UNA cabeza en el [CLS] (incluye V). Misma sonda Ridge + mismos 5 folds que auc_atencion, para los 40 atributos."""
    X = (F - F.mean(0)) / np.clip(F.std(0), 1e-6, None)          # sin √: no son pesos que suman 1, son valores con signo
    pred = np.zeros_like(Y_pm)
    for tr, te in folds:
        r = RidgeCV(alphas=[0.1, 1, 10, 100, 1000, 10000], alpha_per_target=True).fit(X[tr], Y_pm[tr])   # solo 12 variables: alphas más chicos que con 256
        pred[te] = r.predict(X[te])
    return pred

PRED_Z = {}                                       # (variante, semilla, cabeza) -> (N, 40) predicciones
filas = []
for (v, s), z in Z.items():                       # z: (N, 4 cabezas, 12)
    for h in range(N_HEADS):
        pr = auc_salida(z[:, h]); PRED_Z[(v, s, h)] = pr
        aucs = np.array([roc_auc_score(ATTR[:, j], pr[:, j]) for j in range(len(ATTR_NAMES))])
        filas.append({"modelo": v, "semilla": s, "cabeza": h + 1, "AUC Attractive": aucs[IA],
                      "rango de Attractive (1=mejor de 40)": int((aucs > aucs[IA]).sum() + 1),
                      "AUC mediano de los 40": float(np.median(aucs)),
                      **{f"AUC {c}": aucs[ATTR_NAMES.index(c)] for c in CONTEXT}})
AUCT_Z = pd.DataFrame(filas)
AUCT_Z.to_csv(OUTPUT_DIR / "head_output_probe_auc_40attrs.csv", index=False)
display(AUCT_Z.round(3))
cols = [c for c in AUCT_Z.columns if c.startswith("AUC") or c.startswith("rango")]
print("Promedio por modelo (3 semillas × 4 cabezas), sonda sobre la SALIDA de la cabeza:")
display(AUCT_Z.groupby("modelo")[cols].mean().loc[VARIANTS].round(3))

# Comparación directa: mapa de atención (sección 5) vs salida de la cabeza (V incluida)
print("Comparación: AUC medio de la sonda sobre MAPA de atención vs sobre SALIDA de la cabeza")
comp = pd.concat({"mapa de atención": AUCT.groupby("modelo")[cols].mean(),
                  "salida de la cabeza": AUCT_Z.groupby("modelo")[cols].mean()}, axis=1).loc[VARIANTS]
display(comp[[(k, c) for c in ["AUC Attractive", "AUC mediano de los 40", "AUC Eyeglasses", "AUC Male"] for k in ["mapa de atención", "salida de la cabeza"]]].round(3))


## 6. Pregunta 3 — ¿es `Attractive` o son sus correlatos?

Dos controles:

**(a) AUC estratificado.** Divido las imágenes en 4 grupos `Male × Young` y calculo el AUC de `Attractive` **dentro de cada grupo** (con las mismas predicciones de la sección 5), promediando por tamaño de grupo. Si la sonda solo separaba hombres de mujeres, o jóvenes de mayores, dentro del grupo el AUC cae a ~0.5.

**(b) Techo de los correlatos.** ¿Cuánto AUC se logra para `Attractive` usando *solo* las etiquetas `Male`, `Young`, `Heavy_Makeup`, `Wearing_Lipstick`, `Smiling`, `High_Cheekbones` (una regresión logística con CV)? Si ese techo ya es cercano al de la atención, no hay evidencia de nada propio de `Attractive`.

In [ ]:
strat = ATTR[:, ATTR_NAMES.index("Male")] * 2 + ATTR[:, ATTR_NAMES.index("Young")]     # 4 grupos

def auc_estrat(score):
    tot, w = 0.0, 0
    for g in np.unique(strat):
        m = strat == g
        if y[m].min() == y[m].max():
            continue                                    # grupo sin las dos clases
        tot += roc_auc_score(y[m], score[m]) * m.sum(); w += m.sum()
    return tot / w

filas = []
for (v, s, h), pr in PRED.items():
    filas.append({"modelo": v, "semilla": s, "cabeza": h + 1, "AUC global": roc_auc_score(y, pr[:, IA]),
                  "AUC estratificado (Male×Young)": auc_estrat(pr[:, IA])})
EST = pd.DataFrame(filas)
EST["pérdida por estratificar"] = EST["AUC global"] - EST["AUC estratificado (Male×Young)"]
EST.to_csv(OUTPUT_DIR / "attractive_stratified_auc.csv", index=False)
display(EST.groupby("modelo")[["AUC global", "AUC estratificado (Male×Young)", "pérdida por estratificar"]].mean().loc[VARIANTS].round(3))

# AUC del propio modelo (logit de Attractive), global y estratificado: ¿qué tanto es "más que correlatos"?
print("AUC del logit del PROPIO modelo para Attractive:")
display(pd.DataFrame([{"modelo": v, "semilla": s, "global": roc_auc_score(y, LOGIT[(v, s)][:, IA]),
                       "estratificado": auc_estrat(LOGIT[(v, s)][:, IA])} for (v, s) in ATT]).groupby("modelo").mean().drop(columns="semilla").loc[VARIANTS].round(3))

# (b) Techo de los correlatos: regresión logística solo con etiquetas de atributos correlacionados
cor_cols = [ATTR_NAMES.index(c) for c in ["Male", "Young", "Heavy_Makeup", "Wearing_Lipstick", "Smiling", "High_Cheekbones"]]
sc = np.zeros(N)
for tr, te in folds:
    sc[te] = LogisticRegression(max_iter=300).fit(ATTR[tr][:, cor_cols], y[tr]).decision_function(ATTR[te][:, cor_cols])
print(f"\nTecho de los correlatos (solo etiquetas de 6 atributos): AUC = {roc_auc_score(y, sc):.3f} | estratificado Male×Young = {auc_estrat(sc):.3f}")

## 7. Pregunta 4 — ¿qué cabeza, y es consistente entre semillas?

Para cada modelo y semilla, dejo la **cabeza con mayor AUC estratificado** para `Attractive` y la región donde esa cabeza concentra su atención. Como las cabezas no tienen un orden compartido entre semillas, lo que importa es si "la mejor cabeza" mira *siempre la misma región* en las 3 semillas.

In [ ]:
filas = []
for (v, s), a in ATT.items():
    mejor = max(range(N_HEADS), key=lambda h: auc_estrat(PRED[(v, s, h)][:, IA]))
    masa = {k: (a[:, mejor] * REGIONES[k]).sum(1).mean() for k in ["ojos", "nariz", "boca", "fuera de la cara"]}
    top = max(masa, key=masa.get)
    ent = -(a[:, mejor] * np.log(np.clip(a[:, mejor], 1e-8, None))).sum(1)
    filas.append({"modelo": v, "semilla": s, "mejor cabeza": mejor + 1, "AUC estratificado": auc_estrat(PRED[(v, s, mejor)][:, IA]),
                  "región dominante": top, "masa en ella": masa[top], "parches efectivos": float(np.exp(ent).mean())})
MEJOR = pd.DataFrame(filas)
MEJOR.to_csv(OUTPUT_DIR / "attractive_best_head.csv", index=False)
display(MEJOR.round(3))

In [ ]:
# Figura-resumen: AUC (estratificado) de Attractive frente a los atributos de contexto, por variante (promedio sobre semillas y cabezas)
fig, ax = plt.subplots(figsize=(9, 4))
etiquetas = [TARGET + " (estratif.)"] + CONTEXT
anchos = 0.25
for i, v in enumerate(VARIANTS):
    sub = AUCT[AUCT.modelo == v]
    est = EST[EST.modelo == v]["AUC estratificado (Male×Young)"].mean()
    vals = [est] + [sub[f"AUC {c}"].mean() for c in CONTEXT]
    ax.bar(np.arange(len(vals)) + (i - 1) * anchos, vals, width=anchos, label=v)
ax.axhline(0.5, color="k", ls="--", lw=1)
ax.set_xticks(range(len(etiquetas))); ax.set_xticklabels(etiquetas, rotation=20, ha="right")
ax.set_ylabel("AUC de la sonda sobre el mapa de atención"); ax.set_ylim(0.4, 1.0); ax.legend(fontsize=8)
ax.set_title("¿Cuánta información de cada atributo hay en DÓNDE miran las cabezas?\nBarra más alta = el mapa de atención permite predecir mejor el atributo. Línea punteada = azar (0.5).")
plt.tight_layout()
save_fig(fig, "attention_probe_auc_summary", OUTPUT_DIR, tag="celeba", project_root=project_root)
plt.show()

## 8. Etiquetar cada cabeza con la región que mira

Para poder explicar "qué hace" cada cabeza, le pongo un nombre según **dónde concentra su atención**: la región (`ojos`, `nariz`, `boca`, `fuera de la cara` = pelo/fondo) que acumula más masa de atención, siempre que supere el 40 %. Si ninguna región pasa del 40 % **o** la cabeza usa más de 50 parches efectivos, la llamo **`difusa`**. Esta etiqueta se usa en las dos secciones siguientes.

In [ ]:
def etiqueta_region(a_h):
    """a_h: (N, 256) atención de una cabeza. Devuelve (región, masa media en ella, parches efectivos medios)."""
    masa = {k: float((a_h * REGIONES[k]).sum(1).mean()) for k in ["ojos", "nariz", "boca", "fuera de la cara"]}
    top = max(masa, key=masa.get)
    eff = float(np.exp(-(a_h * np.log(np.clip(a_h, 1e-8, None))).sum(1)).mean())
    return (top if masa[top] > 0.40 and eff < 50 else "difusa"), masa[top], eff

REGION_CAB = {(v, s, h): etiqueta_region(ATT[(v, s)][:, h]) for (v, s) in ATT for h in range(N_HEADS)}
pd.DataFrame([{"modelo": v, "semilla": s, "cabeza": h + 1, "región": r[0], "masa en la región top": r[1], "parches efectivos": r[2]}
              for (v, s, h), r in REGION_CAB.items()]).pivot_table(index=["modelo", "semilla"], columns="cabeza", values="región", aggfunc="first").loc[VARIANTS]

## 9. Probing por cabeza: ¿cuánta información de `Attractive` hay en la **salida** de cada cabeza?

Hasta ahora miramos *dónde mira* la cabeza. Aquí miramos **lo que la cabeza entrega**: la salida de cada cabeza en el [CLS] (12 números por imagen). Entreno una **sonda lineal** (regresión logística, 5 folds) por cabeza para predecir `Attractive` y mido el AUC fuera de muestra. También:
- **todas las cabezas juntas** (48 números): el techo de lo que entregan las cuatro;
- **sin la cabeza h** (las otras 3 juntas): cuánto *pierde* el conjunto al quitar h (complementa la ablación de la sección 10);
- AUC **estratificado por Male × Young** (control de correlatos, igual que en la sección 6), y el **techo de correlatos** (0.652 en la sección 6) como referencia.

**Cómo leerlo:** si la sonda de una cabeza da AUC estratificado claramente > 0.652 y > 0.5, esa cabeza **contiene** información de `Attractive` más allá de género y edad. Una cabeza puede *contener* información y aun así el modelo *no usarla*: eso lo responde la ablación.

In [ ]:
def probe_z(F):
    """F (N, d): predicciones fuera de muestra de Attractive con regresión logística sobre F estandarizado."""
    F = (F - F.mean(0)) / np.clip(F.std(0), 1e-6, None)
    sc = np.zeros(N)
    for tr, te in folds:
        sc[te] = LogisticRegression(max_iter=500).fit(F[tr], y[tr]).decision_function(F[te])
    return sc

filas = []
for (v, s), z in Z.items():
    todas = probe_z(z.reshape(N, -1))
    for h in range(N_HEADS):
        solo = probe_z(z[:, h])
        sin = probe_z(np.delete(z, h, axis=1).reshape(N, -1))
        filas.append({"modelo": v, "semilla": s, "cabeza": h + 1, "región": REGION_CAB[(v, s, h)][0],
                      "AUC cabeza sola": roc_auc_score(y, solo), "AUC cabeza sola (estrat.)": auc_estrat(solo),
                      "AUC las 4 juntas": roc_auc_score(y, todas), "AUC las 4 (estrat.)": auc_estrat(todas),
                      "AUC sin esta cabeza": roc_auc_score(y, sin), "pérdida al quitarla (estrat.)": auc_estrat(todas) - auc_estrat(sin)})
PROBE = pd.DataFrame(filas)
PROBE.to_csv(OUTPUT_DIR / "attractive_head_output_probe.csv", index=False)
PROBE.round(3)

In [ ]:
fig, axes = plt.subplots(len(SEEDS), len(VARIANTS), figsize=(14, 8.5), sharey=True)
COL = {"ojos": "tab:red", "boca": "tab:blue", "nariz": "tab:green", "fuera de la cara": "tab:gray", "difusa": "lightgray"}
for c, v in enumerate(VARIANTS):
    for r, s in enumerate(SEEDS):
        ax = axes[r, c]; sub = PROBE[(PROBE.modelo == v) & (PROBE.semilla == s)].sort_values("cabeza")
        ax.bar(range(N_HEADS), sub["AUC cabeza sola (estrat.)"], color=[COL[x] for x in sub["región"]], edgecolor="k")
        ax.axhline(sub["AUC las 4 (estrat.)"].iloc[0], color="k", ls="-", lw=1.5)
        ax.axhline(0.652, color="purple", ls="--", lw=1.2); ax.axhline(0.5, color="gray", ls=":", lw=1)
        ax.set_xticks(range(N_HEADS)); ax.set_xticklabels([f"cab. {h}\n{x}" for h, x in zip(sub["cabeza"], sub["región"])], fontsize=7.5)
        ax.set_ylim(0.45, 0.9); ax.set_title(f"{v} · semilla {s}", fontsize=9)
axes[0, 0].set_ylabel("AUC estratificado Male×Young"); axes[1, 0].set_ylabel("AUC estratificado Male×Young")
handles = [plt.Rectangle((0, 0), 1, 1, color=c, ec="k") for c in COL.values()]
fig.legend(handles + [plt.Line2D([0], [0], color="k"), plt.Line2D([0], [0], color="purple", ls="--"), plt.Line2D([0], [0], color="gray", ls=":")],
           list(COL) + ["las 4 cabezas juntas", "techo de correlatos (0.652)", "azar (0.5)"], loc="lower center", ncol=8, fontsize=8)
fig.suptitle("Probing de la SALIDA de cada cabeza para Attractive (AUC dentro de grupos Male×Young)\n"
             "Color de la barra = región que mira esa cabeza. Barra por encima de la línea morada = la cabeza contiene información de Attractive más allá de género y edad.", fontsize=11)
plt.tight_layout(rect=(0, 0.05, 1, 0.93))
save_fig(fig, "head_output_probe", OUTPUT_DIR, tag="celeba", project_root=project_root)
plt.show()

## 10. Ablación: ¿cuánto **pesa** cada cabeza en la predicción de `Attractive`?

Aquí sí hay causalidad. Para cada cabeza, reemplazo su salida en el [CLS] por su valor medio (la cabeza "se apaga") y vuelvo a calcular la predicción de `Attractive` con el resto del modelo intacto. Defino el **peso** de la cabeza como

> **peso = AUC del modelo completo − AUC con esa cabeza apagada** (sobre las 19 962 imágenes de test; el AUC mide qué tan bien separa `Attractive=1` de `=0`).

- **Peso alto** = al quitarla, el modelo pierde mucha capacidad para predecir `Attractive`: el modelo se apoya en ella.
- **Peso ≈ 0** = el modelo apenas la usa para este atributo.
- Intervalo del 95 % por *bootstrap pareado* (100 remuestreos de las imágenes, mismas imágenes con y sin ablar).
- Para saber si el peso es **específico** de `Attractive` o la cabeza importa para todo, calculo el mismo peso para los otros 39 atributos y reporto su media.
- La tabla también incluye el **cambio medio de la puntuación** (logit) de `Attractive` al apagar la cabeza (positivo = la cabeza empuja hacia "Attractive").

In [ ]:
rng = np.random.default_rng(SEED)
BOOT = [rng.integers(0, N, N) for _ in range(100)]            # mismos remuestreos para todos (bootstrap pareado)

def auc_rapido(lab, sc):
    return roc_auc_score(lab, sc)

filas = []
for (v, s) in ATT:
    full, abl = LOGIT[(v, s)], LABL[(v, s)]
    auc_full = np.array([roc_auc_score(ATTR[:, j], full[:, j]) for j in range(len(ATTR_NAMES))])
    for h in range(N_HEADS):
        auc_abl = np.array([roc_auc_score(ATTR[:, j], abl[h][:, j]) for j in range(len(ATTR_NAMES))])
        peso = auc_full - auc_abl                                          # (40,) peso de la cabeza h para cada atributo
        dif = [roc_auc_score(y[b], full[b, IA]) - roc_auc_score(y[b], abl[h][b, IA]) for b in BOOT]
        reg = REGION_CAB[(v, s, h)]
        filas.append({"modelo": v, "semilla": s, "cabeza": h + 1, "región": reg[0], "parches efectivos": reg[2],
                      "peso en Attractive": peso[IA], "IC95 inf": np.percentile(dif, 2.5), "IC95 sup": np.percentile(dif, 97.5),
                      "peso estrat. (Male×Young)": auc_estrat(full[:, IA]) - auc_estrat(abl[h][:, IA]),
                      "peso medio en los otros 39": float(np.delete(peso, IA).mean()),
                      "Δ puntuación media (completo − sin cabeza)": float((full[:, IA] - abl[h][:, IA]).mean()),
                      "Δ puntuación |media|": float(np.abs(full[:, IA] - abl[h][:, IA]).mean())})
PESO = pd.DataFrame(filas)
PESO.to_csv(OUTPUT_DIR / "attractive_head_weight_ablation.csv", index=False)
PESO.round(4)

In [ ]:
fig, axes = plt.subplots(len(SEEDS), len(VARIANTS), figsize=(14, 8.5), sharey=True)
for c, v in enumerate(VARIANTS):
    for r, s in enumerate(SEEDS):
        ax = axes[r, c]; sub = PESO[(PESO.modelo == v) & (PESO.semilla == s)].sort_values("cabeza")
        err = [sub["peso en Attractive"] - sub["IC95 inf"], sub["IC95 sup"] - sub["peso en Attractive"]]
        ax.bar(range(N_HEADS), sub["peso en Attractive"], yerr=err, capsize=3, color=[COL[x] for x in sub["región"]], edgecolor="k")
        ax.plot(range(N_HEADS), sub["peso medio en los otros 39"], "k_", ms=22, mew=2)
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks(range(N_HEADS)); ax.set_xticklabels([f"cab. {h}\n{x}" for h, x in zip(sub["cabeza"], sub["región"])], fontsize=7.5)
        ax.set_title(f"{v} · semilla {s}", fontsize=9)
axes[1, 0].set_ylabel("peso = AUC completo − AUC sin la cabeza")
handles = [plt.Rectangle((0, 0), 1, 1, color=c, ec="k") for c in COL.values()]
fig.legend(handles + [plt.Line2D([0], [0], color="k", marker="_", ls="", ms=12)], list(COL) + ["peso medio en los otros 39 atributos"], loc="lower center", ncol=6, fontsize=8)
fig.suptitle("PESO de cada cabeza en la predicción de Attractive (ablación)\n"
             "Barra alta = al apagar esa cabeza el modelo pierde mucha capacidad de predecir Attractive. Color = región que mira la cabeza. Raya negra = lo que pesa esa cabeza, en promedio, para los otros atributos.", fontsize=10.5)
plt.tight_layout(rect=(0, 0.05, 1, 0.92))
save_fig(fig, "head_weight_ablation", OUTPUT_DIR, tag="celeba", project_root=project_root)
plt.show()

### 10b. Dos formas de medir el peso: ranking (AUC) vs. puntuación por imagen

El AUC mide si el modelo **ordena bien** las imágenes (Attractive por encima de no-Attractive). Pero una cabeza puede **mover mucho la puntuación** de cada imagen y aun así dejar el orden casi igual (otras cabezas y el flujo residual compensan). Por eso miro también el efecto **por imagen**:

> **efecto de la cabeza h en la imagen i = puntuación del modelo completo − puntuación con h apagada** (en logits; positivo = la cabeza empuja hacia *Attractive*).

Reporto: el efecto medio en imágenes `Attractive=1` y en `=0`; la **alineación** (efecto medio × signo de la etiqueta: positivo = la cabeza empuja hacia la etiqueta correcta); la correlación del efecto con la puntuación final del modelo; y cuánto cae el AUC si se apagan **las 4 cabezas a la vez** (para saber si la atención del bloque 4 en conjunto importa).

In [ ]:
filas = []
signo = np.where(y == 1, 1.0, -1.0)
for (v, s) in ATT:
    full = LOGIT[(v, s)][:, IA]
    auc_all = roc_auc_score(y, full) - roc_auc_score(y, LALL[(v, s)][:, IA])
    for h in range(N_HEADS):
        ef = full - LABL[(v, s)][h][:, IA]                                  # efecto de la cabeza h en cada imagen
        filas.append({"modelo": v, "semilla": s, "cabeza": h + 1, "región": REGION_CAB[(v, s, h)][0],
                      "efecto medio si Attractive=1": ef[y == 1].mean(), "efecto medio si Attractive=0": ef[y == 0].mean(),
                      "alineación con la etiqueta": float((ef * signo).mean()),
                      "correlación efecto~puntuación": float(np.corrcoef(ef, full)[0, 1]),
                      "peso AUC": float(PESO[(PESO.modelo == v) & (PESO.semilla == s) & (PESO.cabeza == h + 1)]["peso en Attractive"].iloc[0]),
                      "caída de AUC al apagar las 4 cabezas": auc_all})
EFECTO_IMG = pd.DataFrame(filas)
EFECTO_IMG.to_csv(OUTPUT_DIR / "attractive_per_image_effect.csv", index=False)
# guardo las puntuaciones de Attractive para poder reutilizarlas sin re-ejecutar los modelos
np.savez_compressed(OUTPUT_DIR / "attractive_logits.npz", y=y, **{f"full__{v}__s{s}": LOGIT[(v, s)][:, IA] for (v, s) in ATT},
                    **{f"abl__{v}__s{s}": LABL[(v, s)][:, :, IA] for (v, s) in ATT}, **{f"ablall__{v}__s{s}": LALL[(v, s)][:, IA] for (v, s) in ATT})
display(EFECTO_IMG.round(3))
print("Caída de AUC al apagar las 4 cabezas del bloque 4 a la vez, por modelo:")
display(EFECTO_IMG.groupby(["modelo", "semilla"])["caída de AUC al apagar las 4 cabezas"].first().round(4))

In [ ]:
fig, axes = plt.subplots(len(SEEDS), len(VARIANTS), figsize=(14, 9), sharey=True)
for c, v in enumerate(VARIANTS):
    for r, s in enumerate(SEEDS):
        ax = axes[r, c]; full = LOGIT[(v, s)][:, IA]
        datos, pos = [], []
        for h in range(N_HEADS):
            ef = full - LABL[(v, s)][h][:, IA]
            datos += [ef[y == 0], ef[y == 1]]; pos += [h * 3, h * 3 + 1]
        bp = ax.boxplot(datos, positions=pos, widths=0.85, showfliers=False, patch_artist=True)
        for k, box in enumerate(bp["boxes"]):
            box.set_facecolor("tab:red" if k % 2 else "tab:blue"); box.set_alpha(0.7)
        ax.axhline(0, color="k", lw=0.8)
        ax.set_xticks([h * 3 + 0.5 for h in range(N_HEADS)])
        ax.set_xticklabels([f"cab. {h + 1}\n{REGION_CAB[(v, s, h)][0]}" for h in range(N_HEADS)], fontsize=7.5)
        ax.set_title(f"{v} · semilla {s}", fontsize=9)
axes[1, 0].set_ylabel("efecto de la cabeza en la puntuación de Attractive\n(puntuación completa − puntuación sin la cabeza)")
fig.legend([plt.Rectangle((0, 0), 1, 1, color="tab:blue", alpha=0.7), plt.Rectangle((0, 0), 1, 1, color="tab:red", alpha=0.7)],
           ["imágenes NO Attractive", "imágenes Attractive"], loc="lower center", ncol=2, fontsize=9)
fig.suptitle("Efecto por imagen de cada cabeza sobre la puntuación de Attractive\n"
             "Caja ROJA por encima de 0 y caja AZUL por debajo de 0 = la cabeza empuja la puntuación hacia la etiqueta correcta (la separa). Cajas cerca de 0 = la cabeza no mueve la puntuación.", fontsize=10.5)
plt.tight_layout(rect=(0, 0.04, 1, 0.92))
save_fig(fig, "head_per_image_effect", OUTPUT_DIR, tag="celeba", project_root=project_root)
plt.show()

## 11. Síntesis: ¿las cabezas que miran los ojos o la boca pesan más?

Agrupo las 36 cabezas (9 modelos × 4) por la región que miran y reporto el peso medio en `Attractive`, el peso medio en los otros atributos y la información de su salida (sonda). **Pocas cabezas por grupo**, así que es descriptivo.

In [ ]:
SINT = PESO.merge(PROBE[["modelo", "semilla", "cabeza", "AUC cabeza sola (estrat.)"]], on=["modelo", "semilla", "cabeza"])
res = SINT.groupby("región").agg(n_cabezas=("cabeza", "size"), peso_Attractive=("peso en Attractive", "mean"),
                                 peso_otros_39=("peso medio en los otros 39", "mean"),
                                 AUC_salida_estrat=("AUC cabeza sola (estrat.)", "mean")).round(3)
res.to_csv(OUTPUT_DIR / "weight_by_region_summary.csv")
display(res)
print("Por variante (solo cabezas NO difusas):")
display(SINT[SINT.región != "difusa"].groupby(["modelo", "región"]).agg(n=("cabeza", "size"), peso_Attractive=("peso en Attractive", "mean"),
                                                                       peso_otros_39=("peso medio en los otros 39", "mean")).round(3))

## 12. Ver una predicción: *por qué* el modelo dice lo que dice

Para un modelo concreto, tomo imágenes de test donde el modelo está **muy seguro** de `Attractive` (arriba) y de **no-Attractive** (abajo) y **acierta**. Cada fila muestra:
- la imagen y la probabilidad que da el modelo;
- las 4 cabezas: su mapa de atención sobre la imagen (más brillante = mira más ahí) y, en el título, **la región que mira** y el **efecto de apagarla en ESTA imagen** (`+` = la cabeza empujaba la puntuación hacia *Attractive*; `−` = hacia *no Attractive*; el número es el cambio de la puntuación del modelo).

Así se lee, imagen por imagen, qué cabeza (y qué zona de la cara) está sosteniendo la decisión.

In [ ]:
def figura_ejemplos(v, s, n=3, semilla_azar=0):
    full, abl = LOGIT[(v, s)][:, IA], LABL[(v, s)][:, :, IA]            # (N,), (4, N)
    p = 1 / (1 + np.exp(-full))
    rng = np.random.default_rng(semilla_azar)
    alto = rng.choice(np.where((y == 1) & (p > 0.9))[0], n, replace=False)
    bajo = rng.choice(np.where((y == 0) & (p < 0.1))[0], n, replace=False)
    idx = list(alto) + list(bajo)
    fig, axes = plt.subplots(len(idx), N_HEADS + 1, figsize=(13, 2.9 * len(idx)))
    for r, i in enumerate(idx):
        axes[r, 0].imshow(X8[i]); axes[r, 0].axis("off")
        axes[r, 0].set_title(f"imagen {i}\netiqueta: {'Attractive' if y[i] else 'no Attractive'}\nmodelo: p(Attractive)={p[i]:.2f}", fontsize=8.5)
        for h in range(N_HEADS):
            ax = axes[r, h + 1]
            mapa = np.kron(ATT[(v, s)][i, h].reshape(GRID, GRID), np.ones((4, 4)))     # 16×16 -> 64×64
            ax.imshow(X8[i]); ax.imshow(mapa, cmap="magma", alpha=0.65, vmin=0, vmax=max(mapa.max(), 1e-6)); ax.axis("off")
            efecto = full[i] - abl[h, i]
            ax.set_title(f"cab. {h + 1} · mira: {REGION_CAB[(v, s, h)][0]}\napagarla cambia la puntuación {efecto:+.2f}", fontsize=8,
                         color="tab:red" if efecto > 0.3 else ("tab:blue" if efecto < -0.3 else "k"))
    fig.suptitle(f"{v}, semilla {s}: qué mira cada cabeza en una imagen y cuánto empuja la predicción de {TARGET}\n"
                 "Brillo = atención. Título rojo: la cabeza empuja hacia Attractive; azul: hacia NO Attractive; negro: efecto pequeño (|cambio| < 0.3).", fontsize=11)
    plt.tight_layout(rect=(0, 0, 1, 0.96))
    return fig

for v, s in [("constrained_rank", 0), ("constrained", 0)]:
    fig = figura_ejemplos(v, s)
    save_fig(fig, f"example_predictions_{v}", OUTPUT_DIR, tag=f"celeba_s{s}", project_root=project_root)
    plt.show()

## 13. Conclusión (2026-11-02; 9 modelos CelebA, test completo, bloque 4)

**Lo que se puede afirmar, en orden de solidez.**

1. **Cada cabeza tiene una región reconocible y la penalización la hace más clara.** Con coseno hay cabezas que miran ojos, boca o pelo/fondo (secciones 4 y 8); sin penalizar, casi todas son difusas. Esto es lo que permite "leer" dónde mira una cabeza.
2. **Hay cabezas que empujan la predicción de `Attractive` en la dirección correcta, y se ve imagen por imagen.** En la figura de ejemplos (sección 12, `constrained_rank` s0) la cabeza que mira los ojos suma +0.7 a +1.1 en imágenes `Attractive` y resta −1.1 a −1.4 en no-Attractive. Es el tipo de explicación que buscabas: "la cabeza de los ojos es la que sostiene esta decisión". También la cabeza de boca empuja en el mismo sentido, con menos fuerza.
3. **Pero, en promedio, el efecto es pequeño.** El efecto medio alineado con la etiqueta es de ~0.1 logits en las cabezas de ojos y de boca, frente a una puntuación que varía ~2 logits (desviación típica). Apagar **una** cabeza baja el AUC como máximo 0.013 (sobre 0.87) y apagar **las cuatro a la vez** lo baja entre 0.001 y 0.030. **La atención del último bloque, en conjunto, apenas sostiene `Attractive`**: la información llega sobre todo por el flujo residual (bloques 1–3) y el MLP. Las cabezas del bloque 4 *ajustan* la puntuación, no la deciden.
3b. **Las cabezas con región facial parecen pesar más, sin que se pueda afirmar.** Alineación media 0.101 (ojos, n=8) y 0.101 (boca, n=4) frente a 0.049 (pelo/fondo, n=14) y 0.075 (difusas, n=9). La diferencia ojos+boca vs. pelo/fondo da p≈0.08 en una prueba de permutación que además **ignora que cabezas del mismo modelo no son independientes**. Es una pista, no un resultado. Y no se repite en todas las semillas: en `constrained` s0 las cabezas de ojos casi no mueven nada (alineación 0.002 y 0.013).
4. **La salida de cada cabeza contiene información de `Attractive`** (AUC de la sonda 0.60–0.81; estratificado Male×Young 0.58–0.74), más que lo que hay en *dónde* mira, pero de forma redundante: las cuatro juntas dan 0.72–0.80 estratificado y quitar una sola cabeza del conjunto casi no cambia nada (pérdida 0.007–0.027).
5. **La penalización no parece "cargar" más peso en una región.** Los pesos por región son parecidos con y sin penalización; lo que cambia es que las cabezas penalizadas son *legibles* (una región, pocos parches), no que pesen más.

**Qué NO se puede decir:** que el modelo "decida `Attractive` por los ojos y la boca". Sí se puede decir que, en algunos modelos, las cabezas de ojos y boca contribuyen de forma consistente con la etiqueta, y que el resto de la decisión sale de otras partes.

**Cuidado con dos cosas.** (a) La ablación por la media apaga la cabeza solo en el [CLS] del bloque 4 (lo único que llega al clasificador); otras formas de ablación (ceros, parchear desde otra imagen) pueden dar números distintos. (b) Las imágenes de ejemplo se eligieron muy seguras (p>0.9 o <0.1), donde los efectos son más grandes que el promedio.

**Siguiente paso que responde la duda abierta:** repetir ablación y sondas en los bloques 1–3 y en el MLP del bloque 4 (¿dónde se *decide* `Attractive`?), y entrenar entropía en CelebA para ver si cambia la legibilidad.